# BIBLIOTECAS

In [ ]:
#import pandas as pd

import joblib as jb
from sklearn.metrics import euclidean_distances

##OCULTAR WARNINGS DO CÓDIGO
import warnings
warnings.filterwarnings("ignore")

# EXECUTANDO MODELO

## CARREGANDO OBJETOS

In [ ]:
## LENDO ARQUIVO
dict_modelo = jb.load('modelo.pkl')
#dict_modelo

In [ ]:
## RETORNANDO AS VARIÁVEIS QUE FORAM UTILIZADAS NO MODELO
colunas_modelo = dict_modelo['VARIAVEIS']
colunas_modelo

In [ ]:
## RETORNANDO BASE DE DADOS ROTULADA NO CLUSTER
#essa base será utilizada para fazer a recomendação
df_filmes = dict_modelo['BASE_DADOS']
df_filmes.head()

## FAZENDO RECOMENDAÇÃO

### OPÇÃO 1 - utilizando somente a tabela já rotulada, **processo simples**


In [ ]:
## SELECIONANDO FILME
display( df_filmes.query("TITLE.str.contains('TOY STORY')") )

In [ ]:
# SELECIONANDO FILME
filme_selecionado = df_filmes.query("MOVIEID == 1 ") #iloc[[0]]
filme_selecionado

In [ ]:
## RETORNANDO O CLUSTER QUE O FILME PERTENCE
cluster_filme_selec  = filme_selecionado.CLUSTER.values[0]
cluster_filme_selec

In [ ]:
## FILTRANDO FILMES DO CLUSTER INDICADO
df_recomendacao = df_filmes.query(f"CLUSTER == {cluster_filme_selec}")
df_recomendacao

#### SELECIONANDO FILMES MAIS "SEMELHANTES"
- É NECESSÁRIO REALIZAR O PROCESSO DE TRANSORMAÇÃO DOS GENEROS EM COLUNA

In [ ]:
## ESTRUTURANDO OS GENEROS DO FILME SELECIONADO NA MESMA ESTRUTURA
# DA TABELA DE TREINAMENTO DO ALGORITMO
genero_filme_selec = filme_selecionado.GENRES.str.get_dummies(sep="|")
genero_filme_selec

In [ ]:
## IGUALANDO A ESTUTURA COM A BASE DE TREINAMENTO
genero_filme_selec = genero_filme_selec.reindex(columns = colunas_modelo, fill_value=0)
genero_filme_selec

In [ ]:
## LISTA PARA ARMAZENAR OS VALORES DE SIMILARIDADE
lista_similaridade = []

## FOR PARA PERCORRER TODOS OS FILMES DO CLUSTER SELECIONADO
for id in df_recomendacao.MOVIEID[:]:
    print(id)

    resultado = df_recomendacao.query(f"MOVIEID == {id}")

    ## ESTRUTURANDO GENERADO NO FORMATO CORRETO
    genero_filme_result = resultado.GENRES.str.get_dummies(sep="|")
    genero_filme_result = genero_filme_result.reindex(columns=colunas_modelo, fill_value=0)
    
    ## CALCULANDO A SIMILARIDADE
    similaridade = euclidean_distances(genero_filme_selec, genero_filme_result)[0][0]

    lista_similaridade.append(similaridade)

In [ ]:
## CRIANDO COLUNA SIMILARIDADE PARA ORDENAR OS TOP N
df_recomendacao['SIMILARIDADE'] = lista_similaridade
df_recomendacao.head(10)

In [ ]:
print("==> VOCÊ QUE ASSISTIU AO FILME: ", filme_selecionado.TITLE.values[0])
for filme in df_recomendacao.sort_values('SIMILARIDADE').TITLE.values[:10]: #.head(10)
    print("  - ",filme)

### OPÇÃO 2 - com o predict do modelo, **processo completo**
- IDEAL PARA QUANDO FOR UM FILME NOVO, QUE NÃO TEM NA BASE DO RESULTADO DO CLUSTER

In [ ]:
## RETORNANDO MODELO(ALGORITMO) PARA FAZER A CLASSIFICAÇÃO EM CADA CLUSTER
modelo = dict_modelo['MODELO']
modelo

In [ ]:
## SELECIONANDO FILME
display( df_filmes.query("TITLE.str.contains('TOY STORY')") )

In [ ]:
# SELECIONANDO FILME
filme_selecionado = df_filmes.query("MOVIEID == 1 ") 
filme_selecionado

In [ ]:
## TRANSFORMANDO FILME SELECIONADO
genero_filme_selec = filme_selecionado.GENRES.str.get_dummies(sep="|")
genero_filme_selec

In [ ]:
## ESTRUTURANDO GENERO FILME SELECIONADO 
# PARA SER EXECUTADO NO MODELO
genero_filme_selec = genero_filme_selec.reindex(columns=colunas_modelo, fill_value=0)

genero_filme_selec

In [ ]:
## CLASSIFICANDO O FILME EM ALGUM DOS CLUSTER
cluster_filme_selec = modelo.predict(genero_filme_selec)[0]
cluster_filme_selec

In [ ]:
## FILTRANDO FILMES DO CLUSTER INDICADO
df_recomendacao = df_filmes.query(f"CLUSTER == {cluster_filme_selec}")
df_recomendacao

#### SELECIONANDO FILMES MAIS "SEMELHANTES"

In [ ]:
## ESTRUTURANDO OS GENEROS DO FILME SELECIONADO NA MESMA ESTRUTURA
# DA TABELA DE TREINAMENTO DO ALGORITMO
genero_filme_selec = filme_selecionado.GENRES.str.get_dummies(sep="|")
genero_filme_selec

In [ ]:
## IGUALANDO A ESTUTURA COM A BASE DE TREINAMENTO
genero_filme_selec = genero_filme_selec.reindex(columns=colunas_modelo, fill_value=0)
genero_filme_selec

In [ ]:
## LISTA PARA ARMAZENAR OS VALORES DE SIMILARIDADE
lista_similaridade = []

## FOR PARA PERCORRER TODOS OS FILMES DO CLUSTER SELECIONADO
for id in df_recomendacao.MOVIEID[:]:
    print(id)

    resultado = df_recomendacao.query(f"MOVIEID == {id}")

    ## ESTRUTURANDO GENERADO NO FORMATO CORRETO
    genero_filme_result = resultado.GENRES.str.get_dummies(sep="|")
    genero_filme_result = genero_filme_result.reindex(columns=colunas_modelo, fill_value=0)
    
    ## CALCULANDO A SIMILARIDADE
    similaridade = euclidean_distances(genero_filme_selec, genero_filme_result)[0][0]

    lista_similaridade.append(similaridade)

In [ ]:
## CRIANDO COLUNA SIMILARIDADE PARA ORDENAR OS TOP N
df_recomendacao['SIMILARIDADE'] = lista_similaridade
df_recomendacao.head(10)

In [ ]:
print("==> VOCÊ QUE ASSISTIU AO FILME: ", filme_selecionado.TITLE.values[0])
for filme in df_recomendacao.sort_values('SIMILARIDADE').TITLE.values[:10]: #.head(10)
    print("  - ",filme)

## FUNÇÃO FINAL

In [ ]:
## FUNÇÃO ENCAPSULANDO TODO PROCEDIMENTO ANTERIOR

def get_recomendacao(id_filme_selec, qtd_recomendacao = 10 ):
    '''
    Função para realizar o processo completo de recomendação de filmes.
    id_filme_selec: filme selecionado pelo usuário
    qtd_recomedacao: quantidade de filmes que vão ser recomendados
    '''
    # PREPARANDO NOVO FILME
    filme_selecionado = df_filmes.query(f"MOVIEID == {id_filme_selec} ")

    ## RETORNANDO O CLUSTER QUE O FILME PERTENCE
    cluster_filme_selec  = filme_selecionado.CLUSTER.values[0]

    ## FILTRANDO FILMES DO CLUSTER INDICADO
    df_recomendacao = df_filmes.query(f"CLUSTER == {cluster_filme_selec} and MOVIEID != {id_filme_selec}")

    ## ESTRUTURANDO OS GENEROS DO FILME SELECIONADO NA MESMA ESTRUTURA
    # DA TABELA DE TREINAMENTO DO ALGORITMO
    genero_filme_selec = filme_selecionado.GENRES.str.get_dummies(sep="|")
    
    ## IGUALANDO A ESTUTURA COM A BASE DE TREINAMENTO
    genero_filme_selec = genero_filme_selec.reindex(columns=colunas_modelo, fill_value=0)
    
    ## LISTA PARA ARMAZENAR OS VALORES DE SIMILARIDADE
    lista_similaridade = []

    ## FOR PARA PERCORRER TODOS OS FILMES DO CLUSTER SELECIONADO
    print("** CALCULANDO SIMILARIDADE DOS FILMES **")
    for id in df_recomendacao.MOVIEID[:]:
        resultado = df_recomendacao.query(f"MOVIEID == {id}")

        ## ESTRUTURANDO GENERADO NO FORMATO CORRETO
        genero_filme_result = resultado.GENRES.str.get_dummies(sep="|")
        genero_filme_result = genero_filme_result.reindex(columns=colunas_modelo, fill_value=0)
        
        ## CALCULANDO A SIMILARIDADE
        similaridade = euclidean_distances(genero_filme_selec, genero_filme_result)[0][0]

        lista_similaridade.append(similaridade)

    ## CRIANDO COLUNA SIMILARIDADE PARA ORDENAR OS TOP N
    df_recomendacao['SIMILARIDADE'] = lista_similaridade

    print("\n\n==> VOCÊ QUE ASSISTIU AO FILME: ", filme_selecionado.TITLE.values[0])
    for filme in df_recomendacao.sort_values('SIMILARIDADE').TITLE.values[:qtd_recomendacao]:
        print("  - ",filme)
    

In [ ]:
## SELECIONANDO FILME
df_filmes.query("TITLE.str.contains('BATMAN')").sort_values('ANO_FILME').head()

In [ ]:
## EXECUTANDO A RECOMENDAÇÃO
get_recomendacao(id_filme_selec = 91054, qtd_recomendacao = 10 )